In [1]:
import numpy as np 
import pandas as pd 
import h5py 
import matplotlib.pyplot as plt 
import seaborn as sns 
import glob
import time 
import torch 
import scipy
from typing import List, Dict, Any, Union
import yaml
import os 
from scipy.stats import ttest_rel, wilcoxon, shapiro

In [2]:
# === Helper: Recursively Load HDF5 into Nested Dict ===
def recursively_load_from_h5(group: Union[h5py.File, h5py.Group]) -> Dict[str, Any]:
    result = {}
    for key, item in group.items():
        if isinstance(item, h5py.Group):
            result[key] = recursively_load_from_h5(item)
        elif isinstance(item, h5py.Dataset):
            result[key] = item[()]  # array or scalar
    for key, val in group.attrs.items():
        result[key] = val
    return result

# === Main Function: Load All Model Entries ===
def load_files_to_dataframe(
    file_list: List[str],
    base_path: str = "/media/ana-caznok/SSD-08/recon-segment/inference_results/"
) -> pd.DataFrame:
    """
    Reads multiple .h5 or .yaml files. Each file may have multiple top-level keys (model names),
    each mapping to a dictionary of metrics/config. Unique models are extracted and merged into a DataFrame.

    Returns:
    - pd.DataFrame with one row per model and fields as columns.
    """
    combined_data = {}  # {model_name: data_dict}

    for filename in file_list:
        ext = os.path.splitext(filename)[1].lower()
        full_path = filename #os.path.join(base_path, filename)

        # Load file content as nested dict
        if ext == ".h5":
            with h5py.File(full_path, "r") as h5file:
                file_data = recursively_load_from_h5(h5file)
        elif ext in [".yaml", ".yml"]:
            with open(full_path, "r") as f:
                file_data = yaml.safe_load(f)
        else:
            raise ValueError(f"Unsupported file format for {filename}")

        # Iterate over all top-level model names in the file
        for model_name, model_info in file_data.items():
            # Only add new models to avoid duplicates
            if model_name not in combined_data:
                combined_data[model_name] = model_info

    # Convert combined data to list of records
    records = []
    for model_name, data in combined_data.items():
        record = {"model_name": model_name}
        record.update(data)
        records.append(record)

    # Create DataFrame
    df = pd.DataFrame(records)
    return df


In [3]:
files = glob.glob('inference_results/*.h5')

In [4]:
df = load_files_to_dataframe(files)

In [5]:
df

,model_name,error,isam_imgs,sam_imgs,ssim_chann,ssim_imgs,dataset,fft,interp,tp
0,restormer_msi2hsi_mraessimsam,"[0.011413793079555035, 0.01222302857786417, 0....","[0.011460542678833008, 0.012092025019228458, 0...","[0.06884612888097763, 0.06432098150253296, 0.0...","[[0.6906588, 0.78292954, 0.8243312, 0.87740594...","[0.9615272, 0.9606634, 0.95301485, 0.9616866, ...",skin,False,False,msi
1,ntire-restormer_rgb2hsi,"[0.048394542187452316, 0.031136609613895416, 0...","[0.03125964477658272, 0.022240852937102318, 0....","[0.09587889909744263, 0.13098526000976562, 0.1...","[[0.9066967, 0.9232098, 0.9274612, 0.9305683, ...","[0.9289908, 0.87427294, 0.9090426, 0.9579202, ...",ntire,False,False,rgb
2,ntire-unet_rgb2fft2ifft2hsi,"[0.12744350731372833, 0.020605672150850296, 0....","[0.040616460144519806, 0.01746373437345028, 0....","[0.14848467707633972, 0.13882163166999817, 0.1...","[[0.91916543, 0.8446351, 0.85987616, 0.8603478...","[0.88294196, 0.92540216, 0.8919406, 0.59796464...",ntire,True,True,rgb2fft2ifft
3,unet_fft2ifft2hsi,"[0.03792598843574524, 0.04261111840605736, 0.0...","[0.02174406498670578, 0.022144321352243423, 0....","[0.15969212353229523, 0.15108025074005127, 0.1...","[[0.95958793, 0.42216203, 0.45497763, 0.738429...","[0.827583, 0.8918211, 0.8535337, 0.80195844, 0...",skin,True,False,fft2ifft
4,unet_fft2ifft2hsi_interp31,"[0.022251417860388756, 0.02503443881869316, 0....","[0.018666289746761322, 0.021042581647634506, 0...","[0.11271315813064575, 0.11249831318855286, 0.1...","[[0.9937429, 0.83221054, 0.813323, 0.9216522, ...","[0.9719799, 0.86072445, 0.95924705, 0.92973787...",skin,True,True,fft2ifft
5,unet_msi2hsi,"[0.028561264276504517, 0.03551485762000084, 0....","[0.023740388453006744, 0.025289276614785194, 0...","[0.14314091205596924, 0.15376773476600647, 0.1...","[[0.11274287, 0.40656546, 0.4089868, 0.5897641...","[0.80122447, 0.75500727, 0.7650121, 0.7631457,...",skin,False,False,msi
6,restormer_fft2ifft2hsi_noclip,"[0.021676672622561455, 0.027255799621343613, 0...","[0.016026856377720833, 0.017839688807725906, 0...","[0.1074339896440506, 0.11124612390995026, 0.12...","[[0.54138625, 0.6925246, 0.7377023, 0.81992215...","[0.9081588, 0.8996011, 0.8872637, 0.89834976, ...",skin,True,False,fft2ifft
7,restormer_fft2ifft2hsi_interp31,"[0.011517148464918137, 0.012467959895730019, 0...","[0.011389877647161484, 0.012215087190270424, 0...","[0.06804566830396652, 0.06483296304941177, 0.0...","[[0.6940012, 0.785322, 0.8236265, 0.8768828, 0...","[0.95916456, 0.9582416, 0.9516537, 0.958004, 0...",skin,True,True,fft2ifft
8,unet_crop_fft2ifft2hsi_interp16,"[0.16391880810260773, 0.1812974065542221, 0.17...","[0.010468291118741035, 0.010449620895087719, 0...","[1.3870809078216553, 1.4030215740203857, 1.389...","[[0.37233147, 0.029831579, 0.09959503, -0.2796...","[0.10448549, 0.07121428, 0.04929682, 0.0745175...",skin,True,True,crop
9,restormer_crop_rgb2hsi,"[0.009554298594594002, 0.009460028260946274, 0...","[0.010815626010298729, 0.011023322120308876, 0...","[0.0740194320678711, 0.06928183138370514, 0.07...","[[0.9962958, 0.7848188, 0.8277453, 0.88160145,...","[0.96561366, 0.967744, 0.9567537, 0.9634454, 0...",skin,False,False,crop


In [6]:
with h5py.File('inference_results/inf_unet_fft2ifft2hsi_interp31.h5', "r") as h5file:
    data = recursively_load_from_h5(h5file)

In [7]:
data.keys()

dict_keys(['unet_fft2ifft2hsi', 'unet_fft2ifft2hsi_interp31', 'unet_msi2hsi'])

In [ ]:
from scipy.stats import ttest_rel, wilcoxon, shapiro

def compare_transformation_effect_auto(
    df: pd.DataFrame,
    model_before: str,
    model_after: str,

    my_col: str = "ssim_imgs",
    alpha: float = 0.05
) -> dict:
    """
    Statistically compares SSIM scores before and after a transformation on the SAME dataset.

    Workflow:
    1. Extract SSIM scores for both models.
    2. Calculate pairwise differences across images.
    3. Test for normality of differences using Shapiro-Wilk.
    4. Choose paired t-test or Wilcoxon test accordingly.
    5. Return p-value, test type, descriptive stats, and significance.

    Parameters:
    ----------
    df : pd.DataFrame
        DataFrame containing rows with model names and a list/array of SSIM scores.
    model_before : str
        The name of the model before the transformation (used to filter the row).
    model_after : str
        The name of the model after the transformation.
    my_col : str
        The name of the column containing SSIM arrays (default is 'ssim_imgs').
    alpha : float
        Significance level for both the normality and hypothesis tests (default is 0.05).

    Returns:
    -------
    dict
        Summary containing test used, p-value, means, and decision.
    """

    # === STEP 1: Retrieve SSIM arrays for both models ===
    # These arrays must be matched: same order, same images
    ssim_before = np.array(df[df["model_name"] == model_before][my_col].values[0])
    ssim_after  = np.array(df[df["model_name"] == model_after][my_col].values[0])

    # === STEP 2: Input validation ===
    assert ssim_before.shape == ssim_after.shape, "SSIM arrays must have the same shape"
    assert ssim_before.ndim == 1, "Each SSIM array must be 1-dimensional (one score per image)"

    # === STEP 3: Compute paired differences ===
    # This gives us the per-image improvement due to the transformation
    diffs = ssim_after - ssim_before

    # === STEP 4: Test if these differences are normally distributed ===
    # Using the Shapiro-Wilk test, which is good for small samples (~<50)
    shapiro_stat, shapiro_p = shapiro(diffs)
    normal = shapiro_p >= alpha  # If p > alpha, we assume normality

    # === STEP 5: Choose test based on normality ===
    if normal:
        # If normal: use the Paired t-test (tests if mean of diffs > 0)
        stat, pval = ttest_rel(ssim_after, ssim_before, alternative="greater")
        test_used = "Paired t-test"
    else:
        # If non-normal: use Wilcoxon signed-rank test (tests if median of diffs > 0)
        # This is a non-parametric alternative that ranks the absolute differences
        stat, pval = wilcoxon(ssim_after, ssim_before, alternative="greater")
        test_used = "Wilcoxon signed-rank test"

    # === STEP 6: Calculate descriptive statistics ===
    mean_before = np.mean(ssim_before)
    mean_after = np.mean(ssim_after)
    mean_diff = np.mean(diffs)

    # === STEP 7: Return a structured summary ===
    return {
        "test_used": test_used,
        "normality_test": {
            "method": "Shapiro-Wilk",
            "statistic": shapiro_stat,
            "p_value": shapiro_p,
            "normal_distribution": normal
        },
        "test_statistic": stat,
        "p_value": pval,
        "mean_before": mean_before,
        "mean_after": mean_after,
        "mean_diff": mean_diff,
        "significant": pval < alpha
    }


In [12]:
df['model_name']

0            restormer_msi2hsi_mraessimsam
1                  ntire-restormer_rgb2hsi
2              ntire-unet_rgb2fft2ifft2hsi
3                        unet_fft2ifft2hsi
4               unet_fft2ifft2hsi_interp31
5                             unet_msi2hsi
6            restormer_fft2ifft2hsi_noclip
7          restormer_fft2ifft2hsi_interp31
8          unet_crop_fft2ifft2hsi_interp16
9                   restormer_crop_rgb2hsi
10                      ntire-unet_rgb2hsi
11        ntire-restormer_rgb2fft2ifft2hsi
12    restormer_crop_fft2ifft2hsi_interp16
Name: model_name, dtype: object

In [26]:
#before and after 
compare = [['restormer_fft2ifft2hsi_noclip', 'restormer_fft2ifft2hsi_interp31'],      #testando a pseudohyper vs interpolacao, o quao importante e a informacao rgb?  
           ['unet_fft2ifft2hsi'            , 'unet_fft2ifft2hsi_interp31'],           # e pra unet?
           ['restormer_msi2hsi_mraessimsam', 'restormer_fft2ifft2hsi_interp31'],      #vale a pena fazer fft ou msi ja eh o suficiente? 
           ['unet_msi2hsi'                 , 'unet_fft2ifft2hsi_interp31'],           #e pra unet? 
           ['restormer_crop_rgb2hsi'       , 'restormer_crop_fft2ifft2hsi_interp16'], #e se considerar a reconstrucao apenas na faixa rgb? 
           ['ntire-unet_rgb2hsi'           , 'ntire-unet_rgb2fft2ifft2hsi'],          #e pra outro dataset? 
           ['ntire-restormer_rgb2hsi'      , 'ntire-restormer_rgb2fft2ifft2hsi']      #e pra outro dataset? 
]

In [27]:
my_columns = ['error', 'sam_imgs', 'isam_imgs', 'ssim_imgs']

In [28]:
# Final output dictionaries
complete_results = {}
results = {}

# Loop over model pairs
for before_model, after_model in compare:
    # Initialize sub-dictionaries
    complete_results.setdefault(after_model, {})[before_model] = {}
    results.setdefault(after_model, {})[before_model] = {}

    # Compare across all metrics
    for col in my_columns:
        # Run statistical comparison with automatic test selection
        summary_stats = compare_transformation_effect_auto(
            df,
            model_before=before_model,
            model_after=after_model,
            my_col=col
        )

        # Store full stats and binary significance flag
        complete_results[after_model][before_model][col] = summary_stats
        results[after_model][before_model][col] = summary_stats['significant']


In [29]:
results

{'restormer_fft2ifft2hsi_interp31': {'restormer_fft2ifft2hsi_noclip': {'error': False,
   'sam_imgs': False,
   'isam_imgs': False,
   'ssim_imgs': True},
  'restormer_msi2hsi_mraessimsam': {'error': False,
   'sam_imgs': False,
   'isam_imgs': False,
   'ssim_imgs': False}},
 'unet_fft2ifft2hsi_interp31': {'unet_fft2ifft2hsi': {'error': False,
   'sam_imgs': False,
   'isam_imgs': False,
   'ssim_imgs': True},
  'unet_msi2hsi': {'error': False,
   'sam_imgs': False,
   'isam_imgs': False,
   'ssim_imgs': True}},
 'restormer_crop_fft2ifft2hsi_interp16': {'restormer_crop_rgb2hsi': {'error': False,
   'sam_imgs': False,
   'isam_imgs': False,
   'ssim_imgs': False}},
 'ntire-unet_rgb2fft2ifft2hsi': {'ntire-unet_rgb2hsi': {'error': False,
   'sam_imgs': False,
   'isam_imgs': False,
   'ssim_imgs': True}},
 'ntire-restormer_rgb2fft2ifft2hsi': {'ntire-restormer_rgb2hsi': {'error': False,
   'sam_imgs': False,
   'isam_imgs': False,
   'ssim_imgs': True}}}